# 04 · Ejercicios — Diseños Fraccionados $2^{k-p}$
(Python)

**Semana 3 — Diseños $2^k$ y fraccionados.**

**Objetivos**
- Construir la estructura de alias para diseños $2^{k-p}$ y verificar la resolución.
- Estimar efectos en fracciones de 8 y 16 corridas usando la gráfica normal.
- Identificar factores activos en experimentos de cribado.
- Analizar estrategias de fold-over y proyección.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels`

> Teoría: [../teoria/03-factoriales-fraccionados.md](../teoria/03-factoriales-fraccionados.md)  
> Equivalente en R: [04-ejercicios-fraccionados_r.ipynb](04-ejercicios-fraccionados_r.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

---
## Ejercicio 1 · Helicóptero de papel ($2^{5-2}$, Resolución III)

Experimento clasico de cribado: se mide el **tiempo de vuelo (s)** de un helicoptero
de papel segun 5 factores con solo 8 corridas ($2^{5-2}$ con generadores $D=AB$, $E=AC$):

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `largo_ala` (A) | corto | largo |
| `ancho_ala` (B) | angosto | ancho |
| `largo_cuerpo` (C) | corto | largo |
| `tipo_papel` (D=AB) | bond | cartulina |
| `clip` (E=AC) | sin clip | con clip |

**Relacion de definicion:** $I = ABD = ACE = BCDE$ — Resolución III.
Cadenas de alias clave: $[A] = A + BD + CE$, $[B] = B + AD$, $[C] = C + AE$.

In [ ]:
df1 = pd.read_csv('../datos/papel-helicoptero-frac.csv')
print('Diseno 2^{5-2} — 8 corridas:')
print(df1.to_string(index=False))

In [ ]:
# Con 8 corridas y 5 factores: solo estimamos efectos principales (7 df totales)
# Ajustamos modelo con los 5 efectos principales
modelo1 = ols('tiempo_vuelo ~ largo_ala + ancho_ala + largo_cuerpo + tipo_papel + clip',
              data=df1).fit()
efectos1 = modelo1.params.drop('Intercept') * 2
nombres1 = {'largo_ala':'A','ancho_ala':'B','largo_cuerpo':'C','tipo_papel':'D','clip':'E'}
efectos1.index = [nombres1.get(x,x) for x in efectos1.index]
print('Efectos estimados (ATENCION: cada uno esta confundido con 2FIs):')
print(efectos1.round(3))

In [ ]:
# Grafica normal de efectos
eff_s1 = efectos1.sort_values()
n1 = len(eff_s1)
z1 = stats.norm.ppf([(i + 0.5)/n1 for i in range(n1)])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(z1, eff_s1.values, s=80, color='steelblue', zorder=3)
for z, e, name in zip(z1, eff_s1.values, eff_s1.index):
    ax.annotate(name, (z, e), xytext=(6, 3), textcoords='offset points', fontsize=11)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Cuantiles normales')
ax.set_ylabel('Efecto estimado (s)')
ax.set_title('Grafica normal de efectos — Helicoptero de papel')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Factores activos: A (largo_ala) y B (ancho_ala).')
print('NOTA: [A]=A+BD+CE, [B]=B+AD+CDE — interpretar con cautela en Res. III.')

---
## Ejercicio 2 · Cribado proceso quimico ($2^{6-2}$, Resolución IV)

Se criban **6 factores** en 16 corridas para maximizar el **rendimiento (%)**.
Diseno $2^{6-2}$ con generadores $E=ABC$, $F=BCD$:

- Relacion de definicion: $I = ABCE = BCDF = ADEF$
- **Resolución IV:** efectos principales **limpios** de 2FIs; las 2FIs estan aliadas entre si.

Alias clave de efectos principales:
$[A]=A+BCDF,\ [B]=B+ACDF,\ [C]=C+ABDF,\ [D]=D+ABCF,\ [E]=E+ABF,\ [F]=F+ADE$

In [ ]:
df2 = pd.read_csv('../datos/proceso-quimico-frac.csv')
print('Diseno 2^{6-2} — 16 corridas:')
print(df2.to_string(index=False))

In [ ]:
# Ajustar modelo con los 6 efectos principales (los estimamos limpios en Res. IV)
modelo2 = ols('rendimiento ~ A + B + C + D + E + F', data=df2).fit()
efectos2 = modelo2.params.drop('Intercept') * 2
print('Efectos principales estimados:')
print(efectos2.round(2))
print(f'\nR2 solo efectos principales = {modelo2.rsquared:.4f}')

In [ ]:
# Grafica normal de efectos
eff_s2 = efectos2.sort_values()
n2 = len(eff_s2)
z2 = stats.norm.ppf([(i + 0.5)/n2 for i in range(n2)])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(z2, eff_s2.values, s=80, color='darkgreen', zorder=3)
for z, e, name in zip(z2, eff_s2.values, eff_s2.index):
    ax.annotate(name, (z, e), xytext=(6, 3), textcoords='offset points', fontsize=11)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Cuantiles normales')
ax.set_ylabel('Efecto estimado (%)')
ax.set_title('Grafica normal de efectos — Proceso quimico (6 factores, 16 corridas)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Modelo reducido con factores activos (A, C, D segun los efectos grandes)
modelo2r = ols('rendimiento ~ A + C + D', data=df2).fit()
anova2r = sm.stats.anova_lm(modelo2r, typ=2)
print('Modelo reducido (A, C, D):')
print(anova2r.round(3))
print(f'R2 = {modelo2r.rsquared:.4f}')
print('\nNOTA: Con Res. IV, 2FIs como AC y CD estan aliadas entre si.')
print('Si se sospecha interaccion, hacer fold-over para desconfundir.')

---
## Ejercicio 3 · Tratamiento de agua ($2^{4-1}$, Resolución IV)

Se estudia la **turbidez residual (NTU)** de agua tratada con cloro.
Media fraccion $2^{4-1}$ con generador $D=ABC$ ($I=ABCD$, Resolución IV):

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `pH` (A) | 6.5 | 8.0 |
| `temperatura` (B) | 10 °C | 25 °C |
| `tiempo_contacto` (C) | 15 min | 30 min |
| `dosis_cloro` (D=ABC) | 1 mg/L | 3 mg/L |

Estructura de alias: $[A]=A+BCD$, $[B]=B+ACD$, $[C]=C+ABD$, $[D]=D+ABC$,
$[AB]=AB+CD$, $[AC]=AC+BD$, $[AD]=AD+BC$.

In [ ]:
df3 = pd.read_csv('../datos/tratamiento-agua-frac.csv')
print('Diseno 2^{4-1} — 8 corridas:')
print(df3.to_string(index=False))

# Verificar D = A*B*C
df3['D_calc'] = df3['pH'] * df3['temperatura'] * df3['tiempo_contacto']
print('\nVerificacion D=ABC:', (df3['dosis_cloro'] == df3['D_calc']).all())
df3.drop(columns='D_calc', inplace=True)

In [ ]:
# Estimacion de efectos principales (limpios de 2FIs en Res. IV)
modelo3 = ols('turbidez ~ pH + temperatura + tiempo_contacto + dosis_cloro', data=df3).fit()
efectos3 = modelo3.params.drop('Intercept') * 2
print('Efectos principales estimados (NTU):')
print(efectos3.sort_values().round(3))

eff_s3 = efectos3.sort_values()
n3 = len(eff_s3)
z3 = stats.norm.ppf([(i + 0.5)/n3 for i in range(n3)])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(z3, eff_s3.values, s=90, color='navy', zorder=3)
for z, e, name in zip(z3, eff_s3.values, eff_s3.index):
    ax.annotate(name, (z, e), xytext=(6, 3), textcoords='offset points', fontsize=11)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Cuantiles normales')
ax.set_ylabel('Efecto (NTU)')
ax.set_title('Grafica normal — Tratamiento de agua')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\nFactores dominantes: dosis_cloro (D, grande negativo) y temperatura (B, positivo).')
print('OBJETIVO: minimizar turbidez -> alta dosis_cloro, alto pH, baja temperatura.')

In [ ]:
# Comparacion de corridas para ilustrar el alias AD=BC
# Con 8 corridas no podemos estimar todas las 2FIs independientemente
print('Aliased 2FIs en este diseno (I=ABCD):')
alias_2fi = {'AB':'CD', 'AC':'BD', 'AD':'BC'}
for k, v in alias_2fi.items():
    print(f'  [{k}] = {k} + {v}')
print('\nPara separar AB de CD seria necesario un fold-over (16 corridas = 2^4 completo).')

# Prediccion de condicion optima
opt = pd.DataFrame({'pH':[1],'temperatura':[-1],'tiempo_contacto':[0],'dosis_cloro':[1]})
pred_opt3 = modelo3.predict(opt)
print(f'\nPrediccion (pH=+1, temp=-1, DC=+1): {pred_opt3.values[0]:.2f} NTU')

---
## Ejercicio 4 · Biofermentacion — produccion de enzimas ($2^{5-1}$, Resolución V)

Se optimiza la **produccion de enzima (U/mL)** en un biofermentador.
Media fraccion de $2^5$ con generador $E=ABCD$ ($I=ABCDE$, **Resolución V**).

Con Resolucion V: **efectos principales e interacciones de 2 factores son estimables
sin confundirse con efectos de orden bajo** (solo se confunden con 3FIs).

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `pH` (A) | 6.0 | 7.5 |
| `temperatura` (B) | 28 °C | 37 °C |
| `oxigeno` (C) | 20 % | 40 % |
| `sustrato` (D) | 10 g/L | 20 g/L |
| `agitacion` (E=ABCD) | 100 rpm | 200 rpm |

In [ ]:
df4 = pd.read_csv('../datos/biofermentacion-frac.csv')
print(f'Diseno 2^{{5-1}} Res. V — {len(df4)} corridas')
print(df4.head(8).to_string(index=False))

# Verificar E = A*B*C*D
df4['E_calc'] = df4['pH'] * df4['temperatura'] * df4['oxigeno'] * df4['sustrato']
print('\nVerificacion E=ABCD:', (df4['agitacion'] == df4['E_calc']).all())
df4.drop(columns='E_calc', inplace=True)

In [ ]:
# Con Res. V podemos estimar TODOS los efectos principales y 2FIs limpiamente
modelo4 = ols(
    'enzima ~ pH + temperatura + oxigeno + sustrato + agitacion '
    '+ pH:temperatura + pH:oxigeno + pH:sustrato + pH:agitacion '
    '+ temperatura:oxigeno + temperatura:sustrato + temperatura:agitacion '
    '+ oxigeno:sustrato + oxigeno:agitacion + sustrato:agitacion',
    data=df4).fit()

efectos4 = modelo4.params.drop('Intercept') * 2
nombres4 = {
    'pH':'A','temperatura':'B','oxigeno':'C','sustrato':'D','agitacion':'E',
    'pH:temperatura':'AB','pH:oxigeno':'AC','pH:sustrato':'AD','pH:agitacion':'AE',
    'temperatura:oxigeno':'BC','temperatura:sustrato':'BD','temperatura:agitacion':'BE',
    'oxigeno:sustrato':'CD','oxigeno:agitacion':'CE','sustrato:agitacion':'DE'
}
efectos4.index = [nombres4.get(x,x) for x in efectos4.index]
print('Efectos estimados (U/mL):')
print(efectos4.abs().sort_values(ascending=False).round(2))

In [ ]:
# Grafica normal de efectos — todos los terminos
eff_s4 = efectos4.sort_values()
n4 = len(eff_s4)
z4 = stats.norm.ppf([(i + 0.5)/n4 for i in range(n4)])

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(z4, eff_s4.values, s=60, color='purple', zorder=3)
for z, e, name in zip(z4, eff_s4.values, eff_s4.index):
    ax.annotate(name, (z, e), xytext=(5, 2), textcoords='offset points', fontsize=9)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Cuantiles normales')
ax.set_ylabel('Efecto estimado (U/mL)')
ax.set_title('Grafica normal de efectos — Biofermentacion (Res. V, 16 corridas)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Modelo reducido con efectos activos: A, B, D, AB
modelo4r = ols('enzima ~ pH + temperatura + sustrato + pH:temperatura', data=df4).fit()
anova4r = sm.stats.anova_lm(modelo4r, typ=2)
print('Modelo reducido (A, B, D, AB):')
print(anova4r.round(3))
print(f'\nR2 = {modelo4r.rsquared:.4f}')

# Comparar: con los mismos datos, proyectar en {A,B,D} — equivale a 2^3 completo
print('\nProyeccion en {A, B, D} — factorial completo 2^3 sobre estos 3 factores:')
print('(Un 2^{5-1} Res. V se proyecta en cualquier subconjunto de 4 factores como 2^4 completo)')

# Diagnosticos finales
resid4 = modelo4r.resid
ajust4 = modelo4r.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid4, line='s', ax=axes[0])
axes[0].set_title('Q-Q residuales - biofermentacion')
axes[1].scatter(ajust4, resid4, alpha=0.8)
axes[1].axhline(0, color='gray', ls='--')
axes[1].set_xlabel('Ajustados (U/mL)')
axes[1].set_ylabel('Residuales')
axes[1].set_title('Residuales vs. ajustados')
plt.tight_layout()
plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid4))